# 04 Cohort Retention

This notebook builds a monthly cohort retention analysis using each user's first observed event month.

## 1. Import libraries and define file paths

We import the libraries we need and set the input and output paths for the cohort analysis files.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_PATH = Path("..") / "outputs" / "events_cleaned.csv"
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COHORT_COUNTS_OUTPUT = OUTPUT_DIR / "cohort_counts.csv"
COHORT_MATRIX_OUTPUT = OUTPUT_DIR / "cohort_retention_matrix.csv"
COHORT_PERCENTAGES_OUTPUT = OUTPUT_DIR / "cohort_retention_percentages.csv"

sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (12, 7)

print(f"Input file: {INPUT_PATH.resolve()}")

## 2. Load the cleaned dataset

We read the cleaned dataset from the `outputs` folder.

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["event_time"])
print("Dataset loaded successfully.")

## 3. Briefly inspect the dataset for cohort work

We only need a few fields for cohort retention: the user identifier and the event month.

In [ ]:
print("Shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print("Event month range:", df["event_month"].min(), "to", df["event_month"].max())
df[["user_id", "event_time", "event_month"]].head()

## 4. Create one monthly activity record per user

For retention analysis, we only need to know whether a user was active in a given month, not how many events they generated.

In [ ]:
user_month_activity = df[["user_id", "event_month"]].drop_duplicates().copy()
print("User-month activity rows:", len(user_month_activity))

## 5. Assign each user to a cohort month

We define the cohort as the month of the user's first observed event in the dataset.

In [ ]:
user_cohorts = (
    user_month_activity.groupby("user_id")["event_month"]
    .min()
    .reset_index()
    .rename(columns={"event_month": "cohort_month"})
)

user_cohorts.head()

## 6. Combine cohort month with user activity month

This lets us compare when a user joined with when the same user returned in later months.

In [ ]:
cohort_data = user_month_activity.merge(user_cohorts, on="user_id", how="left")
cohort_data.head()

## 7. Calculate the month number for each cohort record

Month `0` is the cohort month itself, month `1` is the next month, and so on.

In [ ]:
cohort_data["event_month_period"] = pd.PeriodIndex(cohort_data["event_month"], freq="M")
cohort_data["cohort_month_period"] = pd.PeriodIndex(cohort_data["cohort_month"], freq="M")
cohort_data["cohort_index"] = (
    cohort_data["event_month_period"] - cohort_data["cohort_month_period"]
).apply(lambda x: x.n)

cohort_data[["user_id", "cohort_month", "event_month", "cohort_index"]].head()

## 8. Count active users by cohort month and cohort index

This produces the main cohort counts table.

In [ ]:
cohort_counts = (
    cohort_data.groupby(["cohort_month", "cohort_index"])["user_id"]
    .nunique()
    .reset_index(name="active_users")
    .sort_values(["cohort_month", "cohort_index"])
)

cohort_counts.head(10)

## 9. Build the cohort retention matrix

The retention matrix shows the number of active users for each cohort month across later months.

In [ ]:
cohort_retention_matrix = cohort_counts.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="active_users"
).sort_index()

cohort_retention_matrix

## 10. Calculate retention percentages

We divide each row by the cohort size in month `0` so the table is easier to compare across cohorts.

In [ ]:
cohort_sizes = cohort_retention_matrix[0]
cohort_retention_percentages = cohort_retention_matrix.divide(cohort_sizes, axis=0)
cohort_retention_percentages

## 11. Save the cohort outputs

These CSV files can be reused later for dashboarding and reporting.

In [ ]:
cohort_counts.to_csv(COHORT_COUNTS_OUTPUT, index=False)
cohort_retention_matrix.to_csv(COHORT_MATRIX_OUTPUT)
cohort_retention_percentages.to_csv(COHORT_PERCENTAGES_OUTPUT)

print(f"Saved: {COHORT_COUNTS_OUTPUT.resolve()}")
print(f"Saved: {COHORT_MATRIX_OUTPUT.resolve()}")
print(f"Saved: {COHORT_PERCENTAGES_OUTPUT.resolve()}")

## 12. Visualize retention as a heatmap

This heatmap makes it easier to compare retention patterns across cohort months.

In [ ]:
heatmap_data = cohort_retention_percentages.copy()

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0%",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Retention Rate"},
)
plt.title("Monthly Cohort Retention Heatmap")
plt.xlabel("Months Since First Observed Event")
plt.ylabel("Cohort Month")
plt.tight_layout()
plt.show()

## 13. Print a short summary of the retention findings

This gives a simple text summary of the main retention results.

In [ ]:
first_month_average = cohort_retention_percentages[1].mean()
second_month_average = cohort_retention_percentages[2].mean() if 2 in cohort_retention_percentages.columns else None
largest_cohort_month = cohort_sizes.idxmax()
largest_cohort_size = int(cohort_sizes.max())

print("Key Retention Findings")
print(f"- Number of cohort months analyzed: {cohort_retention_matrix.shape[0]}")
print(f"- Largest cohort month: {largest_cohort_month} ({largest_cohort_size:,} users)")
print(f"- Average month 1 retention: {first_month_average:.2%}")

if second_month_average is not None:
    print(f"- Average month 2 retention: {second_month_average:.2%}")

latest_full_cohort = cohort_retention_percentages.iloc[0]
print("- Cohort table and percentage outputs were saved to the outputs folder.")